# TS pipeline

Fit one spine from mesh and skeleton through basis optimization, radius estimation, and validation.

## Imports

Load MaSCaF fitting tools, SWC plotting, and logging.

In [31]:
import logging
import os
from typing import TYPE_CHECKING

import numpy as np
from scipy.spatial.transform import Rotation

from mascaf import (
    BasisOptimizer,
    FitOptions,
    MeshManager,
    MorphologyGraph,
    SkeletonGraph,
    Validation,
    suggest_fit_parameters,
)
from mascaf.cable_fitting import _compute_morphology_node_radii
from swctools import SWCModel, plot_model

if TYPE_CHECKING:
    import plotly.graph_objects as go  # type: ignore

logging.basicConfig(level=logging.INFO)

print("✅ Libraries imported successfully!")

✅ Libraries imported successfully!


## Per-spine parameters

Camera pose and figure size for each spine index. Maximum edge length and basis-optimizer options come from the fit oracle unless `max_edge_length_override` or `basis_overrides` is set (for example `{"n_rays": 24, "step_scale": 0.25}`). `pdf_scale` is the export resolution for the PDF figures below.

In [32]:
def get_ts_pipeline_params(idx: int) -> dict:
    qst = 0.5
    mcst = 5

    fig_width = 800
    fig_height = 600

    rotations = {
        1: [0, -30, 20],
        2: [-40, 0, 0],
        3: [60, 0, 20],
        4: [-30, 40, 50],
        21: [0, 30, 95],
        24: [0, 0, 0],
        48: [0, 0, 0],
        67: [0, 0, 0],
        76: [0, 0, 0],
    }

    zooms = {
        1: 1.0,
        2: 1.0,
        3: 1.0,
        4: 1.0,
        21: 1.0,
        24: 1.0,
        48: 1.0,
        67: 1.0,
        76: 1.0,
    }

    rot = Rotation.from_euler("xyz", rotations[idx], degrees=True)
    zoom = zooms[idx]
    eye_coord = rot.apply(np.array([1.0, 1.0, 1.0])) * zoom

    return {
        "object_name": f"TS{idx}",
        "qst": qst,
        "mcst": mcst,
        "eye_coord": eye_coord,
        "fig_width": fig_width,
        "fig_height": fig_height,
        "max_edge_length_override": None,
        "basis_overrides": {},
    }

pdf_scale = 2

## Overlap scaling

Branch-overlap constants passed to validation. These are 90% of the perpendicular-cylinder defaults (`C_A = 4`, `C_V = 8/3`).

In [33]:
rescale = 1.0
OVERLAP_SCALING_AREA = 4.0 * rescale
OVERLAP_SCALING_VOLUME = 8.0 / 3.0 * rescale

## Load mesh and skeleton

Load the processed mesh and MCF skeleton for the chosen spine, then take `max_edge_length` and basis-optimizer options from the fit oracle. The figures are the mesh alone and the mesh with the skeleton, written under `viz/`.

In [34]:
spine_idx = 1
params = get_ts_pipeline_params(spine_idx)

mesh_path = f"../data/mesh/processed/{params['object_name']}.obj"
mm = MeshManager(mesh_path=mesh_path)
model_length = mm.bounding_box_diagonal()

polylines_name = f"TS{spine_idx}_qst{params['qst']}_mcst{params['mcst']}"
skeleton = SkeletonGraph.from_txt(
    f"../data/mcf_skeletons/{polylines_name}.polylines.txt"
)

suggested = suggest_fit_parameters(
    mm,
    skeleton,
    overrides=params.get("basis_overrides") or None,
)
for line in suggested.rationale:
    print(f"Oracle: {line}")

if params.get("max_edge_length_override") is not None:
    params["max_edge_length"] = float(params["max_edge_length_override"])
else:
    params["max_edge_length"] = float(suggested.max_edge_length)
params["optimizer_options"] = suggested.basis_optimizer_options
params["max_edge_length_fraction"] = (
    params["max_edge_length"] / model_length if model_length > 0 else float("nan")
)
params["max_edge_length_tag"] = int(round(float(params["max_edge_length"])))
print(
    f"Using mel={params['max_edge_length']:.4g} "
    f"(frac={params['max_edge_length_fraction']:.4g}, "
    f"mel/t={suggested.mel_over_thickness:.4g}, "
    f"tag={params['max_edge_length_tag']})"
)

fig_out_dir = f"../viz/ts{spine_idx}"
os.makedirs(fig_out_dir, exist_ok=True)

mesh_fig: "go.Figure" = mm.visualize_mesh_3d(skel=None, show_axes=False, title="")

eye_coord = params["eye_coord"]

mesh_fig.update_layout(
    scene=dict(
        camera=dict(
            eye={"x": eye_coord[0], "y": eye_coord[1], "z": eye_coord[2]},
            projection=dict(type="perspective"),
        ),
        aspectmode="data",
    )
)
mesh_fig.write_image(
    f"{fig_out_dir}/TS{spine_idx}_mesh.pdf",
    format="pdf",
    engine="kaleido",
    width=600,
    height=450,
    scale=pdf_scale,
)
mesh_fig.show()

mesh_skel_fig: "go.Figure" = mm.visualize_mesh_3d(
    skel=skeleton, show_axes=False, title=""
)
mesh_skel_fig.update_layout(
    scene=dict(
        camera=dict(
            eye={"x": eye_coord[0], "y": eye_coord[1], "z": eye_coord[2]},
            projection=dict(type="perspective"),
        ),
        aspectmode="data",
    )
)
mesh_skel_fig.write_image(
    f"{fig_out_dir}/TS{spine_idx}_mesh_skel.pdf",
    format="pdf",
    engine="kaleido",
    width=600,
    height=450,
    scale=pdf_scale,
)
mesh_skel_fig.show()

INFO:mascaf.mesh:Loaded mesh: 2378 vertices, 4788 faces
INFO:choreographer.utils._tmpfile:TemporaryDirectory.cleanup() worked.
INFO:choreographer.utils._tmpfile:shutil.rmtree worked.


INFO:mascaf.fit_oracle:Fit oracle: mel=133.768 (frac=0.04935, mel/t=1.716), n_rays=12, active_resample=[0.02468, 0.04935]×D
C:\Users\MainUser\AppData\Local\Temp\ipykernel_36840\3026965432.py:53: DeprecationWarning: 
Support for the 'engine' argument is deprecated and will be removed after September 2025.
Kaleido will be the only supported engine at that time.

  mesh_fig.write_image(
INFO:choreographer.browsers.chromium:Chromium init'ed with kwargs {}
INFO:choreographer.browsers.chromium:Found chromium path: C:\Program Files (x86)\Google\Chrome\Application\chrome.exe
INFO:choreographer.utils._tmpfile:Temp directory created: C:\Users\MainUser\AppData\Local\Temp\tmpg1t5k4do.
INFO:choreographer.browser_async:Opening browser.
INFO:choreographer.utils._tmpfile:Temp directory created: C:\Users\MainUser\AppData\Local\Temp\tmpafs7rivs.
INFO:choreographer.browsers.chromium:Temporary directory at: C:\Users\MainUser\AppData\Local\Temp\tmpafs7rivs


Oracle: thickness-based mel = 2 × SDF median (77.9434) → 155.887
Oracle: capped by skeleton sampling density: L/n_target = 8828.7/66 → 133.768 (mel/t=1.716)
Oracle: equivalent bbox fraction = 0.04935 (D=2710.45, D/t=34.77, L/t=113.3)
Using mel=133.8 (frac=0.04935, mel/t=1.716, tag=134)


INFO:kaleido.kaleido:Conforming 1 to file:///C:/Users/MainUser/AppData/Local/Temp/tmpg1t5k4do/index.html
INFO:kaleido.kaleido:Getting tab from queue (has 1)
INFO:kaleido.kaleido:Got 0C90
INFO:kaleido.kaleido:Reloading tab 0C90 before return.
INFO:kaleido.kaleido:Putting tab 0C90 back (queue size: 0).
INFO:kaleido.kaleido:Waiting for all cleanups to finish.
INFO:kaleido.kaleido:Exiting Kaleido.
INFO:choreographer.utils._tmpfile:TemporaryDirectory.cleanup() worked.
INFO:choreographer.utils._tmpfile:shutil.rmtree worked.
INFO:choreographer.browser_async:Closing browser.
INFO:choreographer.utils._tmpfile:TemporaryDirectory.cleanup() worked.
INFO:choreographer.utils._tmpfile:shutil.rmtree worked.
INFO:choreographer.browser_async:Closing browser.
INFO:kaleido.kaleido:Cancelling tasks.
INFO:kaleido.kaleido:Exiting Kaleido/Choreo.
INFO:choreographer.utils._tmpfile:TemporaryDirectory.cleanup() worked.
INFO:choreographer.utils._tmpfile:shutil.rmtree worked.
INFO:kaleido.kaleido:Cancelling tasks.

C:\Users\MainUser\AppData\Local\Temp\ipykernel_36840\3026965432.py:75: DeprecationWarning: 
Support for the 'engine' argument is deprecated and will be removed after September 2025.
Kaleido will be the only supported engine at that time.

  mesh_skel_fig.write_image(
INFO:choreographer.browsers.chromium:Chromium init'ed with kwargs {}
INFO:choreographer.browsers.chromium:Found chromium path: C:\Program Files (x86)\Google\Chrome\Application\chrome.exe
INFO:choreographer.utils._tmpfile:Temp directory created: C:\Users\MainUser\AppData\Local\Temp\tmpcu9dj48e.
INFO:choreographer.browser_async:Opening browser.
INFO:choreographer.utils._tmpfile:Temp directory created: C:\Users\MainUser\AppData\Local\Temp\tmprcxdqkgb.
INFO:choreographer.browsers.chromium:Temporary directory at: C:\Users\MainUser\AppData\Local\Temp\tmprcxdqkgb
INFO:kaleido.kaleido:Conforming 1 to file:///C:/Users/MainUser/AppData/Local/Temp/tmpcu9dj48e/index.html
INFO:kaleido.kaleido:Getting tab from queue (has 1)
INFO:kaleido

## Basis optimization

Resample the skeleton into a morphology basis and move nodes with `BasisOptimizer` before any radius fitting. The figure overlays the original basis (red) and the optimized basis (blue).

In [35]:
basis = MorphologyGraph.from_skeleton_graph_resample(
    skeleton,
    float(params["max_edge_length"]),
)
print(
    f"Initial basis: {basis.number_of_nodes()} nodes, "
    f"{basis.number_of_edges()} edges"
)

optimizer = BasisOptimizer(basis, mm.mesh, params["optimizer_options"])
optimized_basis = optimizer.optimize()
stats = optimizer.get_optimization_stats()
print("Basis optimization statistics:")
for key, value in stats.items():
    print(f"  {key}: {value}")

basis_opt_fig: "go.Figure" = mm.visualize_mesh_3d(
    skel=[basis, optimized_basis],
    show_axes=False,
    title="",
    skel_color=["red", "blue"],
    skel_line_width=3.0,
    skel_marker_size=2.0,
)
basis_opt_fig.update_layout(
    scene=dict(
        camera=dict(
            eye={"x": eye_coord[0], "y": eye_coord[1], "z": eye_coord[2]},
            projection=dict(type="perspective"),
        ),
        aspectmode="data",
    )
)
basis_opt_fig.write_image(
    f"{fig_out_dir}/TS{spine_idx}_mel{params['max_edge_length_tag']}_basis_opt.pdf",
    format="pdf",
    engine="kaleido",
    width=600,
    height=450,
    scale=pdf_scale,
)
basis_opt_fig.show()

INFO:mascaf.morphology_graph:Resampled morphology: 88 nodes, 94 edges (max_edge_length=133.7682, source 376 nodes / 383 edges, 49 sections)
INFO:mascaf.basis_optimizer:Starting basis optimization...
INFO:mascaf.basis_optimizer:  Input: 88 nodes, 94 edges, total length 8563.1565
INFO:mascaf.basis_optimizer:Phase 0 - Pruning
INFO:mascaf.basis_optimizer:  Removing terminal stubs with length < 85.7676
INFO:mascaf.basis_optimizer:  Pruning done: 88 → 79 nodes (1 rounds)
INFO:mascaf.basis_optimizer:Phase 1 - Snapping: 6 nodes outside mesh


Initial basis: 88 nodes, 94 edges


INFO:mascaf.basis_optimizer:Phase 2 - Forcing: max 30 iterations
INFO:mascaf.basis_optimizer:  Forcing alpha_s=0.1000 step_scale=0.5000 ray_jitter=0.1000 active_resample=True
INFO:mascaf.basis_optimizer:  Active resample edge lengths in [6.688411e+01, 1.337682e+02] (fractions of bbox diagonal)
INFO:mascaf.basis_optimizer:  Iteration 0: active resample merged=3 split=0
INFO:mascaf.basis_optimizer:  Iteration 0: avg movement = 4.482027, centering_error = 8.199123e+02
INFO:mascaf.basis_optimizer:  Iteration 1: active resample merged=1 split=1
INFO:mascaf.basis_optimizer:  Iteration 1: avg movement = 3.666115, centering_error = 6.514212e+02
INFO:mascaf.basis_optimizer:  Iteration 2: active resample merged=0 split=1
INFO:mascaf.basis_optimizer:  Iteration 2: avg movement = 3.239450, centering_error = 5.678210e+02
INFO:mascaf.basis_optimizer:  Iteration 3: active resample merged=0 split=2
INFO:mascaf.basis_optimizer:  Iteration 3: avg movement = 2.902492, centering_error = 5.200313e+02
INFO:

Basis optimization statistics:
  num_nodes: 85
  num_edges: 91
  num_terminal_nodes: 9
  num_branch_nodes: 14
  total_length: 8731.26736306491
  nodes_outside_mesh: 0


INFO:kaleido.kaleido:Conforming 1 to file:///C:/Users/MainUser/AppData/Local/Temp/tmpjddaxt2i/index.html
INFO:kaleido.kaleido:Getting tab from queue (has 1)
INFO:kaleido.kaleido:Got 5AEB
INFO:kaleido.kaleido:Reloading tab 5AEB before return.
INFO:kaleido.kaleido:Putting tab 5AEB back (queue size: 0).
INFO:kaleido.kaleido:Waiting for all cleanups to finish.
INFO:kaleido.kaleido:Exiting Kaleido.
INFO:choreographer.utils._tmpfile:TemporaryDirectory.cleanup() worked.
INFO:choreographer.utils._tmpfile:shutil.rmtree worked.
INFO:choreographer.browser_async:Closing browser.
INFO:choreographer.utils._tmpfile:TemporaryDirectory.cleanup() worked.
INFO:choreographer.utils._tmpfile:shutil.rmtree worked.
INFO:choreographer.browser_async:Closing browser.
INFO:kaleido.kaleido:Cancelling tasks.
INFO:kaleido.kaleido:Exiting Kaleido/Choreo.
INFO:choreographer.utils._tmpfile:TemporaryDirectory.cleanup() worked.
INFO:choreographer.utils._tmpfile:shutil.rmtree worked.
INFO:kaleido.kaleido:Cancelling tasks.

## Cable fitting

Estimate node radii on the optimized basis with the equivalent-area strategy, write the SWC. The figure is the fitted morphology.

In [36]:
swc_out_dir = f"../data/swc/current/{polylines_name}"
swc_filepath = f"{swc_out_dir}/TS{spine_idx}_mel{params['max_edge_length_tag']}.swc"

if not os.path.exists(swc_out_dir):
    os.makedirs(swc_out_dir)

fit_options = FitOptions(
    max_edge_length=params["max_edge_length"],
    radius_strategy="equivalent_area",
    section_probe_eps=1e-4,
    section_probe_tries=3,
    multi_tangent_reduction="mean",
    basis_optimizer_options=None,
)
morph = optimized_basis.copy()
_compute_morphology_node_radii(morph, mm.mesh, fit_options)

morph.to_swc_file(swc_filepath)

model = SWCModel.from_swc_file(swc_filepath)
model.print_attributes(node_info=False, edge_info=False)
title = f"TS{spine_idx}_mel{params['max_edge_length_tag']}"
morph_fig = plot_model(
    swc_model=model,
    slider=False,
    title="",
    width=800,
    height=600,
    show_axes=False,
)
morph_fig.update_layout(
    scene=dict(
        camera=dict(
            eye={"x": eye_coord[0], "y": eye_coord[1], "z": eye_coord[2]},
            projection=dict(type="perspective"),
        ),
        aspectmode="data",
    )
)
morph_filename = (
    f"{fig_out_dir}/TS{spine_idx}_mel{params['max_edge_length_tag']}_" f"morph.pdf"
)
morph_fig.write_image(
    morph_filename,
    format="pdf",
    engine="kaleido",
    width=600,
    height=450,
    scale=pdf_scale,
)
morph_fig.show()

INFO:mascaf.cable_fitting:Computing node radii: strategy=equivalent_area, 85 nodes, probe_eps=1.00e-04, tries=3
INFO:choreographer.utils._tmpfile:TemporaryDirectory.cleanup() worked.
INFO:choreographer.utils._tmpfile:shutil.rmtree worked.
INFO:mascaf.cable_fitting:Radii computed: min=13.908073 max=154.524242 mean=62.551667 median=56.142560
INFO:swctools.io:parse_swc start strict=True validate_reconnections=True float_tol=1e-09
INFO:swctools.io:parse_swc done records=92 reconnections=7 header=11
INFO:swctools.model:SWCModel.from_parse_result records=92 reconnections=7 header=11
INFO:swctools.model:SWCModel.from_swc_file built nodes=92 edges=91 strict=True validate_reconnections=True
INFO:swctools.geometry:batch_frusta count=91 sides=16 end_caps=False verts=2912 faces=2912
INFO:swctools.geometry:FrustaSet.from_swc_model edges=91 sides=16 end_caps=False
INFO:swctools.viz:plot_model slider=False frusta=91 show_frusta=True show_centroid=True
C:\Users\MainUser\AppData\Local\Temp\ipykernel_36

SWCModel: nodes=92, edges=91, components=1, cycles=0, branch_points=11, roots=1, leaves=15, self_loops=0, density=0.0217


INFO:kaleido.kaleido:Conforming 1 to file:///C:/Users/MainUser/AppData/Local/Temp/tmpn9sjpc20/index.html
INFO:kaleido.kaleido:Getting tab from queue (has 1)
INFO:kaleido.kaleido:Got E17E
INFO:kaleido.kaleido:Reloading tab E17E before return.
INFO:kaleido.kaleido:Putting tab E17E back (queue size: 0).
INFO:kaleido.kaleido:Waiting for all cleanups to finish.
INFO:kaleido.kaleido:Exiting Kaleido.
INFO:choreographer.utils._tmpfile:TemporaryDirectory.cleanup() worked.
INFO:choreographer.utils._tmpfile:shutil.rmtree worked.
INFO:choreographer.browser_async:Closing browser.
INFO:choreographer.utils._tmpfile:TemporaryDirectory.cleanup() worked.
INFO:choreographer.utils._tmpfile:shutil.rmtree worked.
INFO:choreographer.browser_async:Closing browser.
INFO:kaleido.kaleido:Cancelling tasks.
INFO:kaleido.kaleido:Exiting Kaleido/Choreo.
INFO:choreographer.utils._tmpfile:TemporaryDirectory.cleanup() worked.
INFO:choreographer.utils._tmpfile:shutil.rmtree worked.
INFO:kaleido.kaleido:Cancelling tasks.

### Validation without area fit

In [37]:
validator = Validation(mm, skeleton, morph)
validator.full_validation(
    overlap_scaling_area=OVERLAP_SCALING_AREA,
    overlap_scaling_volume=OVERLAP_SCALING_VOLUME,
)

INFO:mascaf.validation:Initialized Validation from MorphologyGraph
INFO:mascaf.validation:  Mesh: 2378 vertices, 4788 faces
INFO:mascaf.validation:  Skeleton: 376 nodes, 383 edges
INFO:mascaf.validation:  MorphologyGraph: 85 nodes, 91 edges
INFO:mascaf.validation:Validation Results, account_for_overlaps=False:
INFO:mascaf.validation:-- Volume Comparison:
INFO:mascaf.validation:---- Mesh volume:       70083740.2341
INFO:mascaf.validation:---- Morphology volume: 146492536.5279
INFO:mascaf.validation:---- Ratio:             2.0902
INFO:mascaf.validation:---- Error:             76408796.2938
INFO:mascaf.validation:---- Relative error:    109.02%
INFO:mascaf.validation:-- Surface Area Comparison:
INFO:mascaf.validation:---- Mesh area:         3124839.6923
INFO:mascaf.validation:---- Morphology area:   4115638.4670
INFO:mascaf.validation:---- Ratio:             1.3171
INFO:mascaf.validation:---- Error:             990798.7748
INFO:mascaf.validation:---- Relative error:    31.71%
INFO:mascaf.

## Surface-area normalization

Scale radii so the morphology surface area matches the mesh, without overlap correction.

In [38]:
morph.scale_radii_to_match_mesh(
    mm.mesh, metric="surface_area", account_for_overlaps=False
)

swc_filepath = f"{swc_out_dir}/TS{spine_idx}_mel{params['max_edge_length_tag']}_norm.swc"
morph.to_swc_file(swc_filepath)

swc_model = SWCModel.from_swc_file(swc_filepath)
swc_model.print_attributes(node_info=False, edge_info=False)
title = f"TS{spine_idx}_s{params['max_edge_length_tag']}"
norm_fig: "go.Figure" = plot_model(
    swc_model=swc_model,
    slider=False,
    title="",
    width=800,
    height=600,
    show_axes=False,
)
norm_fig.update_layout(
    scene=dict(
        camera=dict(
            eye={"x": eye_coord[0], "y": eye_coord[1], "z": eye_coord[2]},
            projection=dict(type="perspective"),
        ),
        aspectmode="data",
    )
)
norm_filename = (
    f"{fig_out_dir}/TS{spine_idx}_mel{params['max_edge_length_tag']}_" f"morph_norm.pdf"
)
norm_fig.write_image(
    norm_filename,
    format="pdf",
    engine="kaleido",
    width=600,
    height=450,
    scale=pdf_scale,
)
norm_fig.show()

print(swc_filepath)

INFO:swctools.io:parse_swc start strict=True validate_reconnections=True float_tol=1e-09
INFO:swctools.io:parse_swc done records=92 reconnections=7 header=11
INFO:swctools.model:SWCModel.from_parse_result records=92 reconnections=7 header=11
INFO:swctools.model:SWCModel.from_swc_file built nodes=92 edges=91 strict=True validate_reconnections=True
INFO:swctools.geometry:batch_frusta count=91 sides=16 end_caps=False verts=2912 faces=2912
INFO:swctools.geometry:FrustaSet.from_swc_model edges=91 sides=16 end_caps=False
INFO:choreographer.utils._tmpfile:TemporaryDirectory.cleanup() worked.
INFO:choreographer.utils._tmpfile:shutil.rmtree worked.
INFO:swctools.viz:plot_model slider=False frusta=91 show_frusta=True show_centroid=True
C:\Users\MainUser\AppData\Local\Temp\ipykernel_36840\1358501362.py:31: DeprecationWarning: 
Support for the 'engine' argument is deprecated and will be removed after September 2025.
Kaleido will be the only supported engine at that time.

  norm_fig.write_image(
I

SWCModel: nodes=92, edges=91, components=1, cycles=0, branch_points=11, roots=1, leaves=15, self_loops=0, density=0.0217


INFO:kaleido.kaleido:Conforming 1 to file:///C:/Users/MainUser/AppData/Local/Temp/tmpqz0hdqas/index.html
INFO:kaleido.kaleido:Getting tab from queue (has 1)
INFO:kaleido.kaleido:Got 6E9F
INFO:kaleido.kaleido:Reloading tab 6E9F before return.
INFO:kaleido.kaleido:Putting tab 6E9F back (queue size: 0).
INFO:kaleido.kaleido:Waiting for all cleanups to finish.
INFO:kaleido.kaleido:Exiting Kaleido.
INFO:choreographer.utils._tmpfile:TemporaryDirectory.cleanup() worked.
INFO:choreographer.utils._tmpfile:shutil.rmtree worked.
INFO:choreographer.browser_async:Closing browser.
INFO:choreographer.utils._tmpfile:TemporaryDirectory.cleanup() worked.
INFO:choreographer.utils._tmpfile:shutil.rmtree worked.
INFO:choreographer.browser_async:Closing browser.
INFO:kaleido.kaleido:Cancelling tasks.
INFO:kaleido.kaleido:Exiting Kaleido/Choreo.
INFO:choreographer.utils._tmpfile:TemporaryDirectory.cleanup() worked.
INFO:choreographer.utils._tmpfile:shutil.rmtree worked.
INFO:kaleido.kaleido:Cancelling tasks.

../data/swc/current/TS1_qst0.5_mcst5/TS1_mel134_norm.swc


### Validation with area fit

In [39]:
validator = Validation(mm, skeleton, morph)
validator.full_validation(
    overlap_scaling_area=OVERLAP_SCALING_AREA,
    overlap_scaling_volume=OVERLAP_SCALING_VOLUME,
)

INFO:mascaf.validation:Initialized Validation from MorphologyGraph
INFO:mascaf.validation:  Mesh: 2378 vertices, 4788 faces
INFO:mascaf.validation:  Skeleton: 376 nodes, 383 edges
INFO:mascaf.validation:  MorphologyGraph: 85 nodes, 91 edges
INFO:mascaf.validation:Validation Results, account_for_overlaps=False:
INFO:mascaf.validation:-- Volume Comparison:
INFO:mascaf.validation:---- Mesh volume:       70083740.2341
INFO:mascaf.validation:---- Morphology volume: 90123135.3012
INFO:mascaf.validation:---- Ratio:             1.2859
INFO:mascaf.validation:---- Error:             20039395.0671
INFO:mascaf.validation:---- Relative error:    28.59%
INFO:mascaf.validation:-- Surface Area Comparison:
INFO:mascaf.validation:---- Mesh area:         3124839.6923
INFO:mascaf.validation:---- Morphology area:   3124839.6923
INFO:mascaf.validation:---- Ratio:             1.0000
INFO:mascaf.validation:---- Error:             -0.0000
INFO:mascaf.validation:---- Relative error:    -0.00%
INFO:mascaf.valida

## Morphology versus skeleton

Overlay the normalized morphology (translucent) with the original skeleton points.

In [40]:
skel_pointset = skeleton.to_point_set()

vs_fig: "go.Figure" = plot_model(
    swc_model=swc_model,
    opacity=0.2,
    title="",
    width=800,
    height=600,
    show_axes=False,
    point_set=skel_pointset,
    point_color="red",
    point_size=model_length * 0.0015,
)
vs_fig.update_layout(
    scene=dict(
        camera=dict(
            eye={"x": eye_coord[0], "y": eye_coord[1], "z": eye_coord[2]},
            projection=dict(type="perspective"),
        ),
        aspectmode="data",
    )
)
vs_filename = (
    f"{fig_out_dir}/TS{spine_idx}_mel{params['max_edge_length_tag']}_"
    f"morph_vs_skel.pdf"
)
vs_fig.write_image(
    vs_filename,
    format="pdf",
    engine="kaleido",
    width=600,
    height=450,
    scale=pdf_scale,
)
vs_fig.show()

INFO:choreographer.utils._tmpfile:TemporaryDirectory.cleanup() worked.
INFO:choreographer.utils._tmpfile:shutil.rmtree worked.
INFO:swctools.geometry:batch_spheres count=-1 stacks=6 slices=12 verts=23312 faces=45120
INFO:swctools.geometry:PointSet.from_points n=376 base_radius=1.0 stacks=6 slices=12
INFO:swctools.geometry:batch_frusta count=91 sides=16 end_caps=False verts=2912 faces=2912
INFO:swctools.geometry:FrustaSet.from_swc_model edges=91 sides=16 end_caps=False
INFO:swctools.geometry:batch_spheres count=376 stacks=6 slices=12 verts=23312 faces=45120
INFO:swctools.geometry:PointSet.scaled radius_scale=4.065675620614607
INFO:swctools.viz:plot_model slider=False frusta=91 show_frusta=True show_centroid=True
C:\Users\MainUser\AppData\Local\Temp\ipykernel_36840\2276369519.py:27: DeprecationWarning: 
Support for the 'engine' argument is deprecated and will be removed after September 2025.
Kaleido will be the only supported engine at that time.

  vs_fig.write_image(
INFO:choreographer